# Phase 1: Denoising Pretraining

Heterogeneous MoE encoder-decoder trained on curated literary corpus.

**Setup:** Run cells 1-3 once per session. Cell 3 needs the corpus data — either from Drive or upload.

**Training:** Cell 4 runs training. Checkpoints save to Drive automatically.

In [ ]:
#@title 1. Clone repo + install deps
import os

# Clone
if not os.path.exists('/content/corpus-training'):
    !git clone https://github.com/s0ph1stry/corpus-training.git /content/corpus-training
else:
    !cd /content/corpus-training && git pull

# Install tokenizers (torch is pre-installed on Colab)
!pip install -q tokenizers wandb

os.chdir('/content/corpus-training')
print('Ready.')

In [ ]:
#@title 2. Mount Google Drive for corpus data + checkpoints
from google.colab import drive
drive.mount('/content/drive')

DRIVE_BASE = '/content/drive/MyDrive/corpus-training'
os.makedirs(DRIVE_BASE, exist_ok=True)
os.makedirs(f'{DRIVE_BASE}/checkpoints', exist_ok=True)
print(f'Drive mounted. Data dir: {DRIVE_BASE}')

In [ ]:
#@title 3. Sync corpus data
#
# First time: upload cleaned/ as a zip to Drive/corpus-training/cleaned.zip
# After that, this cell handles everything.
#
import shutil

CLEANED_DIR = '/content/corpus-training/cleaned'
DRIVE_CLEANED_ZIP = f'{DRIVE_BASE}/cleaned.zip'
DRIVE_CLEANED_DIR = f'{DRIVE_BASE}/cleaned'

if os.path.exists(CLEANED_DIR) and len(os.listdir(CLEANED_DIR)) > 100:
    print(f'Corpus already present: {len(os.listdir(CLEANED_DIR))} files')
elif os.path.exists(DRIVE_CLEANED_ZIP):
    print('Extracting corpus from Drive zip...')
    !unzip -q {DRIVE_CLEANED_ZIP} -d /content/corpus-training/
    print(f'Extracted: {len(os.listdir(CLEANED_DIR))} files')
elif os.path.exists(DRIVE_CLEANED_DIR):
    print('Copying corpus from Drive...')
    shutil.copytree(DRIVE_CLEANED_DIR, CLEANED_DIR)
    print(f'Copied: {len(os.listdir(CLEANED_DIR))} files')
else:
    print('ERROR: No corpus data found!')
    print(f'Upload cleaned.zip to: {DRIVE_CLEANED_ZIP}')
    print('Or copy the cleaned/ directory to: {DRIVE_CLEANED_DIR}')
    raise FileNotFoundError('Corpus data not found on Drive')

# Symlink checkpoints to Drive for persistence
CKPT_DIR = '/content/corpus-training/checkpoints'
DRIVE_CKPT = f'{DRIVE_BASE}/checkpoints'
if os.path.exists(CKPT_DIR) and not os.path.islink(CKPT_DIR):
    # Move any existing checkpoints to Drive
    for f in os.listdir(CKPT_DIR):
        src = os.path.join(CKPT_DIR, f)
        if os.path.isfile(src):
            shutil.move(src, DRIVE_CKPT)
    shutil.rmtree(CKPT_DIR)
if not os.path.exists(CKPT_DIR):
    os.symlink(DRIVE_CKPT, CKPT_DIR)
    print(f'Checkpoints symlinked to Drive')

# Quick sanity check
!python -c "from data.dataset import CorpusDataset; d = CorpusDataset('.', samples_per_epoch=10); print(f'Dataset OK: {len(d.texts)} texts')"

In [ ]:
#@title 4. Train Phase 1 — Tiny
#@markdown ---
#@markdown **Preset:** `tiny` (8.95M params) or `small` (162M params)
PRESET = 'tiny' #@param ['tiny', 'small']
#@markdown **Steps:** total training steps
TOTAL_STEPS = 1000 #@param {type: 'integer'}
#@markdown **Batch size:** 32 for tiny, 16 for small on A100
BATCH_SIZE = 32 #@param {type: 'integer'}
#@markdown **Resume from checkpoint:**
RESUME = '' #@param {type: 'string'}
#@markdown **Use wandb logging:**
USE_WANDB = False #@param {type: 'boolean'}

cmd = f'python -m training.train_phase1 --preset {PRESET} --total-steps {TOTAL_STEPS} --batch-size {BATCH_SIZE}'
if RESUME:
    cmd += f' --resume {RESUME}'
if not USE_WANDB:
    cmd += ' --no-wandb'

print(f'Running: {cmd}\n')
!{cmd}

In [ ]:
#@title 5. Check training results
import torch
import glob

# Find latest checkpoint
ckpts = sorted(glob.glob('checkpoints/phase1/step_*.pt'))
if ckpts:
    latest = ckpts[-1]
    ckpt = torch.load(latest, map_location='cpu', weights_only=False)
    print(f'Latest checkpoint: {latest}')
    print(f'  Step: {ckpt["global_step"]}')
    if 'loss_history' in ckpt:
        losses = ckpt['loss_history']
        print(f'  First loss: {losses[0]:.4f}')
        print(f'  Last loss:  {losses[-1]:.4f}')
        print(f'  Reduction:  {(1 - losses[-1]/losses[0])*100:.1f}%')
    if 'routing_entropy' in ckpt:
        print(f'  Routing entropy: {ckpt["routing_entropy"]}')
else:
    print('No checkpoints found.')

In [ ]:
#@title 6. GPU info
!nvidia-smi
import torch
print(f'\nPyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')